In [165]:
import fitz
import pandas as pd
import re
import os

files = {
    "Овощи": "./Цены на овощи и фрукты/овощи.pdf",
    "Фрукты": "./Цены на овощи и фрукты/фрукты.pdf"
}

pdf_data = {}

for name, path in files.items():
    doc = fitz.open(path)

    pages = []
    for page in doc:
        pages.append(page.get_text())

    pdf_data[name] = {
        "pages": pages,
        "text": "\n".join(pages),
        "page_count": len(doc)
    }

    print(f"{name}:")
    print(f"Количество страниц: {len(doc)}")
    print(f"Количество символов: {len(pdf_data[name]['text'])}")
    print(f"Количество строк: {len(pdf_data[name]['text'].splitlines())}")
    print()

Овощи:
Количество страниц: 7
Количество символов: 27495
Количество строк: 344

Фрукты:
Количество страниц: 6
Количество символов: 22228
Количество строк: 269



In [166]:
for name, data in pdf_data.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)

    print(data["text"][:2500])


Овощи
New York Terminal Market Vegetables Prices (NX_FV020)
November 06, 2026
Agricultural Marketing Service
Specialty Crops Market News
Page 1
NX_FV020
Email us with accessibility issues with this report.
HERBS
---ANISE:  MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24.00-26.00 24s 22.00-24.00 fine appearance 26.00-30.00 mostly 28.00 
---ARUGULA:  MARKET STEADY. cartons bunched NEW JERSEY 24s 22.00-24.00 with root 4/5 bushel crates bunched FLORIDA 24s 24.00 
with root cartons 2 1-1/2 lb film bags FLORIDA BABY TYPE 16.00-17.00 
---BASIL:  OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA air 12.00-14.00 bushel cartons bunched FLORIDA WITH ROOT 15s 36.00-
42.00 mostly 38.00-40.00 
---BAY LEAVES:  OFFERINGS LIGHT. repacked enroute 22.00-23.00 1 KG cartons film bag loose 1 lb film bags loose repacked enroute 
14.00 
---CELERIAC (CELERY ROOT):  MARKET ABOUT STEADY. 25 lb cartons MICHIGAN 20.00 20 lb cartons CANADA QUEBEC 15.00-16.00 
---CHIVES:  MARKET STEADY. 1 kg cartons bunched COLOMBIA gr

In [167]:
for name, data in pdf_data.items():
    lines = data["text"].splitlines()

    product_lines = [
        line.strip()
        for line in lines
        if line.strip().startswith("---")
    ]

    product_names = []

    for line in product_lines:
        match = re.match(r"---([^:]+):", line)

        if match:
            product_names.append(match.group(1).strip())

    print(f"\n{name}")
    print(f"Количество товарных блоков: {len(product_lines)}")
    print(f"Количество уникальных названий: {len(set(product_names))}")
    print(f"Количество повторяющихся названий: "
          f"{len(product_names) - len(set(product_names))}")


Овощи
Количество товарных блоков: 125
Количество уникальных названий: 124
Количество повторяющихся названий: 1

Фрукты
Количество товарных блоков: 57
Количество уникальных названий: 53
Количество повторяющихся названий: 4


In [168]:
price_pattern = re.compile(
    r"(?<!\d)\d+\.\d{2}(?:\s*-\s*\d+\.\d{2})?"
)

for name, data in pdf_data.items():
    lines = data["text"].splitlines()

    lines_with_prices = [
        line.strip()
        for line in lines
        if price_pattern.search(line)
    ]

    print(f"\n{name}")
    print(f"Строк с потенциальными ценовыми значениями: "
          f"{len(lines_with_prices)}")

    print("\nПримеры:")
    for line in lines_with_prices[:10]:
        print(line)


Овощи
Строк с потенциальными ценовыми значениями: 250

Примеры:
---ANISE:  MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24.00-26.00 24s 22.00-24.00 fine appearance 26.00-30.00 mostly 28.00
---ARUGULA:  MARKET STEADY. cartons bunched NEW JERSEY 24s 22.00-24.00 with root 4/5 bushel crates bunched FLORIDA 24s 24.00
with root cartons 2 1-1/2 lb film bags FLORIDA BABY TYPE 16.00-17.00
---BASIL:  OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA air 12.00-14.00 bushel cartons bunched FLORIDA WITH ROOT 15s 36.00-
42.00 mostly 38.00-40.00
---BAY LEAVES:  OFFERINGS LIGHT. repacked enroute 22.00-23.00 1 KG cartons film bag loose 1 lb film bags loose repacked enroute
14.00
---CELERIAC (CELERY ROOT):  MARKET ABOUT STEADY. 25 lb cartons MICHIGAN 20.00 20 lb cartons CANADA QUEBEC 15.00-16.00
---CHIVES:  MARKET STEADY. 1 kg cartons bunched COLOMBIA greenhouse 12s 17.00-18.00 1 lb film bags loose NONE repacked local
greenhouse 10.00-11.00

Фрукты
Строк с потенциальными ценовыми значениями: 181

Примеры:


In [169]:
import re
import pandas as pd

price_pattern = re.compile(r"(?<!\d)\d+\.\d{2}(?:\s*-\s*\d+\.\d{2})?")

for name, data in pdf_data.items():
    text = data["text"]

    # Разделяем текст на товарные блоки
    blocks = re.split(r"(?=^---)", text, flags=re.MULTILINE)
    blocks = [b.strip() for b in blocks if b.strip().startswith("---")]

    results = []

    for block in blocks:
        title_match = re.match(r"---([^:]+):", block)
        product = title_match.group(1).strip() if title_match else "Не определено"

        results.append({
            "Продукт": product,
            "Есть числовые значения": bool(price_pattern.search(block)),
            "Есть NONE": "NONE" in block.upper(),
            "Текст блока": block
        })

    result_df = pd.DataFrame(results)

    print(f"\n{name}")
    print("Всего товарных блоков:", len(result_df))
    print("Блоков с числовыми значениями:",
          result_df["Есть числовые значения"].sum())
    print("Блоков без числовых значений:",
          (~result_df["Есть числовые значения"]).sum())
    print("Блоков с обозначением NONE:",
          result_df["Есть NONE"].sum())

    print("\nБлоки без числовых значений:")
    display(
        result_df.loc[
            ~result_df["Есть числовые значения"],
            ["Продукт", "Текст блока"]
        ]
    )


Овощи
Всего товарных блоков: 125
Блоков с числовыми значениями: 125
Блоков без числовых значений: 0
Блоков с обозначением NONE: 24

Блоки без числовых значений:


,Продукт,Текст блока



Фрукты
Всего товарных блоков: 57
Блоков с числовыми значениями: 57
Блоков без числовых значений: 0
Блоков с обозначением NONE: 1

Блоки без числовых значений:


,Продукт,Текст блока


In [170]:
units = [
    "kg", "lb", "oz", "bushel", "carton",
    "crate", "bag", "sack", "flat", "box",
    "container", "jar"
]

for name, data in pdf_data.items():
    text = data["text"].lower()

    print(f"\n{name}")
    for unit in units:
        count = len(re.findall(rf"\b{unit}\b", text))
        if count > 0:
            print(f"{unit}: {count}")


Овощи
kg: 26
lb: 88
oz: 13
bushel: 57
carton: 1
bag: 6
flat: 3
container: 3
jar: 6

Фрукты
kg: 10
lb: 49
oz: 10
bushel: 12
carton: 1
flat: 1
container: 3


In [171]:
for name, data in pdf_data.items():
    blocks = re.split(
        r"(?=^---)", data["text"], flags=re.MULTILINE
    )
    blocks = [b.strip() for b in blocks if b.strip().startswith("---")]

    products = []

    for block in blocks:
        match = re.match(r"---([^:]+):", block)
        if match:
            products.append(match.group(1).strip())

    duplicates = pd.Series(products)
    duplicate_names = duplicates[duplicates.duplicated(keep=False)].unique()

    print(f"\n{name}")
    print("Повторяющиеся наименования:")

    for product in duplicate_names:
        print(f"\n{product}")
        for block in blocks:
            if block.startswith(f"---{product}:"):
                print(block)


Овощи
Повторяющиеся наименования:

GINGER ROOT
---GINGER ROOT:  MARKET STEADY. 30 lb cartons BRAZIL 32.00-33.00 fine appearance 34.00-36.00 CHINA 24.00-28.00 mostly 26.00 fine 
appearance 30.00-34.00 mostly 32.00 5 lb cartons NONE repacked local 8.00 10 lb cartons NONE repacked local 16.00
---GINGER ROOT:  MARKET STEADY. 30 lb cartons PERU 26.00-32.00 mostly 28.00-30.00 fair appearance holdovers 22.00-24.00 fine 
appearance 34.00-38.00 mostly 36.00 
USDA, AMS, Specialty Crops Market News
Bronx, NY 10474
Phone (718) 542-2225 | https://mymarketnews.ams.usda.gov/
Source:

Фрукты
Повторяющиеся наименования:

BLACKBERRIES
---BLACKBERRIES:  MARKET SLIGHTLY LOWER. flats 12 6-oz cups with lids CALIFORNIA large fine appearance 34.00 MEXICO large 20.00-
22.00
---BLACKBERRIES:  MARKET SLIGHTLY LOWER. flats 12 6-oz cups with lids CALIFORNIA large 28.00-30.00 fair appearance holdovers 12.00-
16.00 mostly 14.00 MEXICO large fair appearance holdovers 18.00

RASPBERRIES
---RASPBERRIES:  MARKET SLIGHT

In [172]:
market_rows = []

for name, data in pdf_data.items():
    for page_num, page_text in enumerate(data["pages"], start=1):
        for line_num, line in enumerate(page_text.splitlines(), start=1):
            clean_line = re.sub(r"\s+", " ", line).strip()

            if clean_line:
                market_rows.append({
                    "Dataset": name,
                    "Page": page_num,
                    "LineNumber": line_num,
                    "RawText": line,
                    "CleanText": clean_line
                })

df_market = pd.DataFrame(market_rows)

print("Размер очищенного набора:", df_market.shape)
display(df_market.head(20))

Размер очищенного набора: (602, 5)


,Dataset,Page,LineNumber,RawText,CleanText
0,Овощи,1,1,New York Terminal Market Vegetables Prices (NX...,New York Terminal Market Vegetables Prices (NX...
1,Овощи,1,2,"November 06, 2026","November 06, 2026"
2,Овощи,1,3,Agricultural Marketing Service,Agricultural Marketing Service
3,Овощи,1,4,Specialty Crops Market News,Specialty Crops Market News
4,Овощи,1,5,Page 1,Page 1
5,Овощи,1,6,NX_FV020,NX_FV020
6,Овощи,1,7,Email us with accessibility issues with this r...,Email us with accessibility issues with this r...
7,Овощи,1,8,HERBS,HERBS
8,Овощи,1,9,---ANISE: MARKET ABOUT STEADY. cartons CALIFO...,---ANISE: MARKET ABOUT STEADY. cartons CALIFOR...
9,Овощи,1,10,---ARUGULA: MARKET STEADY. cartons bunched NE...,---ARUGULA: MARKET STEADY. cartons bunched NEW...


In [173]:
import re
import pandas as pd

market_records = []

# Служебные строки, которые не относятся к описаниям товаров
service_patterns = [
    r"^Agricultural Marketing Service$",
    r"^Specialty Crops Market News$",
    r"^Page \d+$",
    r"^NX_[A-Z0-9]+$",
    r"^Email us with accessibility issues",
]

for (dataset, page), group in df_market.groupby(["Dataset", "Page"], sort=False):

    lines = group["CleanText"].tolist()

    report_date = None
    current_section = None
    current_product = None
    current_description = []

    def save_record():
        if current_product is not None:
            market_records.append({
                "Dataset": dataset,
                "Page": page,
                "ReportDate": report_date,
                "Section": current_section,
                "Product": current_product,
                "Description": " ".join(current_description)
            })

    for line in lines:

        # Дата отчёта
        date_match = re.search(
            r"\b([A-Z][a-z]+ \d{1,2}, \d{4})\b", line
        )

        if date_match:
            report_date = pd.to_datetime(
                date_match.group(1),
                format="%B %d, %Y"
            )
            continue

        # Удаляем служебные строки
        if any(re.search(pattern, line, re.IGNORECASE)
               for pattern in service_patterns):
            continue

        # Заголовок раздела, например HERBS
        if (
            line.isupper()
            and len(line) < 50
            and not line.startswith("---")
            and not re.search(r"\d", line)
        ):
            save_record()
            current_product = None
            current_description = []
            current_section = line
            continue

        # Начало новой записи о товаре
        product_match = re.match(
            r"^\s*---\s*(.+?):\s*(.*)$", line
        )

        if product_match:
            save_record()

            current_product = product_match.group(1).strip()
            current_description = []

            first_description = product_match.group(2).strip()

            if first_description:
                current_description.append(first_description)

            continue

        # Продолжение описания текущего товара
        if current_product is not None:
            current_description.append(line)

    save_record()

df_market_records = pd.DataFrame(market_records)

print("Количество восстановленных записей:", len(df_market_records))
display(df_market_records.head(15))

Количество восстановленных записей: 182


,Dataset,Page,ReportDate,Section,Product,Description
0,Овощи,1,2026-11-06,HERBS,ANISE,MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24...
1,Овощи,1,2026-11-06,HERBS,ARUGULA,MARKET STEADY. cartons bunched NEW JERSEY 24s ...
2,Овощи,1,2026-11-06,HERBS,BASIL,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...
3,Овощи,1,2026-11-06,HERBS,BAY LEAVES,OFFERINGS LIGHT. repacked enroute 22.00-23.00 ...
4,Овощи,1,2026-11-06,HERBS,CELERIAC (CELERY ROOT),MARKET ABOUT STEADY. 25 lb cartons MICHIGAN 20...
5,Овощи,1,2026-11-06,HERBS,CHIVES,MARKET STEADY. 1 kg cartons bunched COLOMBIA g...
6,Овощи,1,2026-11-06,HERBS,CILANTRO,MARKET STEADY. 4/5 bushel crates bunched FLORI...
7,Овощи,1,2026-11-06,HERBS,CIPOLINOS,OFFERINGS VERY LIGHT. NONE repacked local and ...
8,Овощи,1,2026-11-06,HERBS,DILL,MARKET STEADY. 1/2 bushel cartons bunched NEW ...
9,Овощи,1,2026-11-06,HERBS,DRY ESCHALLOT,OFFERINGS LIGHT. 10 lb sacks NEW YORK ORANGE C...


In [174]:
for _, row in df_market_records.head(10).iterrows():
    print(f"Товар: {row['Product']}")
    print(f"Раздел: {row['Section']}")
    print(f"Дата: {row['ReportDate']}")
    print(f"Описание: {row['Description']}")
    print("-" * 80)

Товар: ANISE
Раздел: HERBS
Дата: 2026-11-06 00:00:00
Описание: MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24.00-26.00 24s 22.00-24.00 fine appearance 26.00-30.00 mostly 28.00
--------------------------------------------------------------------------------
Товар: ARUGULA
Раздел: HERBS
Дата: 2026-11-06 00:00:00
Описание: MARKET STEADY. cartons bunched NEW JERSEY 24s 22.00-24.00 with root 4/5 bushel crates bunched FLORIDA 24s 24.00 with root cartons 2 1-1/2 lb film bags FLORIDA BABY TYPE 16.00-17.00
--------------------------------------------------------------------------------
Товар: BASIL
Раздел: HERBS
Дата: 2026-11-06 00:00:00
Описание: OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA air 12.00-14.00 bushel cartons bunched FLORIDA WITH ROOT 15s 36.00- 42.00 mostly 38.00-40.00
--------------------------------------------------------------------------------
Товар: BAY LEAVES
Раздел: HERBS
Дата: 2026-11-06 00:00:00
Описание: OFFERINGS LIGHT. repacked enroute 22.00-23.00 1 KG cartons film 

In [175]:
price_pattern = re.compile(
    r"(?<![\d.])(\d{1,3}\.\d{2})(?:\s*-\s*(\d{1,3}\.\d{2}))?"
)
price_records = []

for _, row in df_market_records.iterrows():

    description = row["Description"]
    matches = list(price_pattern.finditer(description))

    offers = []

    for match in matches:

        price_min = float(match.group(1))
        price_max = (
            float(match.group(2))
            if match.group(2)
            else price_min
        )

        context = description[:match.start()]

        # Проверяем, является ли цена уточнением mostly
        is_mostly = bool(
            re.search(r"\bmostly\s*$", context, re.IGNORECASE)
        )

        if is_mostly and offers:
            offers[-1]["MostlyMin"] = price_min
            offers[-1]["MostlyMax"] = price_max
            continue

        # Контекст от предыдущей цены до текущей
        previous_end = (
            matches[matches.index(match) - 1].end()
            if matches.index(match) > 0
            else 0
        )

        offer_context = description[previous_end:match.start()].strip()

        offers.append({
            "Dataset": row["Dataset"],
            "Page": row["Page"],
            "ReportDate": row["ReportDate"],
            "Section": row["Section"],
            "Product": row["Product"],
            "OfferNumber": len(offers) + 1,
            "OfferContext": offer_context,
            "PriceMin": price_min,
            "PriceMax": price_max,
            "MostlyMin": None,
            "MostlyMax": None,
            "PriceType": (
                "Диапазон" if match.group(2)
                else "Одиночная цена"
            ),
            "OriginalDescription": description
        })

    price_records.extend(offers)

df_market_prices = pd.DataFrame(price_records)

print("Количество ценовых предложений:", len(df_market_prices))
print("Уникальных товаров:", df_market_prices["Product"].nunique())

display(df_market_prices.head(20))

Количество ценовых предложений: 968
Уникальных товаров: 177


,Dataset,Page,ReportDate,Section,Product,OfferNumber,OfferContext,PriceMin,PriceMax,MostlyMin,MostlyMax,PriceType,OriginalDescription
0,Овощи,1,2026-11-06,HERBS,ANISE,1,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,24.0,26.0,NaN,NaN,Диапазон,MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24...
1,Овощи,1,2026-11-06,HERBS,ANISE,2,24s,22.0,24.0,NaN,NaN,Диапазон,MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24...
2,Овощи,1,2026-11-06,HERBS,ANISE,3,fine appearance,26.0,30.0,28.0,28.0,Диапазон,MARKET ABOUT STEADY. cartons CALIFORNIA 18s 24...
3,Овощи,1,2026-11-06,HERBS,ARUGULA,1,MARKET STEADY. cartons bunched NEW JERSEY 24s,22.0,24.0,NaN,NaN,Диапазон,MARKET STEADY. cartons bunched NEW JERSEY 24s ...
4,Овощи,1,2026-11-06,HERBS,ARUGULA,2,with root 4/5 bushel crates bunched FLORIDA 24s,24.0,24.0,NaN,NaN,Одиночная цена,MARKET STEADY. cartons bunched NEW JERSEY 24s ...
5,Овощи,1,2026-11-06,HERBS,ARUGULA,3,with root cartons 2 1-1/2 lb film bags FLORIDA...,16.0,17.0,NaN,NaN,Диапазон,MARKET STEADY. cartons bunched NEW JERSEY 24s ...
6,Овощи,1,2026-11-06,HERBS,BASIL,1,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...,12.0,14.0,NaN,NaN,Диапазон,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...
7,Овощи,1,2026-11-06,HERBS,BASIL,2,bushel cartons bunched FLORIDA WITH ROOT 15s,36.0,42.0,38.0,40.0,Диапазон,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...
8,Овощи,1,2026-11-06,HERBS,BAY LEAVES,1,OFFERINGS LIGHT. repacked enroute,22.0,23.0,NaN,NaN,Диапазон,OFFERINGS LIGHT. repacked enroute 22.00-23.00 ...
9,Овощи,1,2026-11-06,HERBS,BAY LEAVES,2,1 KG cartons film bag loose 1 lb film bags loo...,14.0,14.0,NaN,NaN,Одиночная цена,OFFERINGS LIGHT. repacked enroute 22.00-23.00 ...


In [176]:
products_with_prices = set(
    zip(
        df_market_prices["Dataset"],
        df_market_prices["Page"],
        df_market_prices["Product"]
    )
)

products_without_prices = df_market_records[
    ~df_market_records.apply(
        lambda row: (
            row["Dataset"],
            row["Page"],
            row["Product"]
        ) in products_with_prices,
        axis=1
    )
]

print("Записей без найденных цен:", len(products_without_prices))
display(products_without_prices[
    ["Dataset", "Page", "Product", "Description"]
])

Записей без найденных цен: 0


,Dataset,Page,Product,Description


In [177]:
for product in ["ANISE", "BASIL", "BAY LEAVES"]:

    print(f"\n{product}")

    display(
        df_market_prices[
            df_market_prices["Product"] == product
        ][[
            "OfferContext",
            "PriceMin",
            "PriceMax",
            "MostlyMin",
            "MostlyMax"
        ]]
    )


ANISE


,OfferContext,PriceMin,PriceMax,MostlyMin,MostlyMax
0,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,24.0,26.0,NaN,NaN
1,24s,22.0,24.0,NaN,NaN
2,fine appearance,26.0,30.0,28.0,28.0



BASIL


,OfferContext,PriceMin,PriceMax,MostlyMin,MostlyMax
6,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...,12.0,14.0,NaN,NaN
7,bushel cartons bunched FLORIDA WITH ROOT 15s,36.0,42.0,38.0,40.0



BAY LEAVES


,OfferContext,PriceMin,PriceMax,MostlyMin,MostlyMax
8,OFFERINGS LIGHT. repacked enroute,22.0,23.0,NaN,NaN
9,1 KG cartons film bag loose 1 lb film bags loo...,14.0,14.0,NaN,NaN


In [178]:
packaging_patterns = [
    "cartons",
    "crates",
    "sacks",
    "bags",
    "bushel",
    "film bags",
    "plastic containers",
    "flats",
    "baskets",
    "jars"
]

for pattern in packaging_patterns:
    count = df_market_prices["OfferContext"].str.contains(
        pattern, case=False, na=False
    ).sum()

    print(f"{pattern}: {count}")

cartons: 245
crates: 29
sacks: 12
bags: 34
bushel: 69
film bags: 32
plastic containers: 1
flats: 16
baskets: 3
jars: 3


In [179]:
def extract_packaging(text):
    text = text.lower()

    patterns = {
        "cartons": r"\bcartons?\b",
        "crates": r"\bcrates?\b",
        "bags": r"\bbags?\b",
        "sacks": r"\bsacks?\b",
        "flats": r"\bflats?\b",
        "baskets": r"\bbaskets?\b",
        "jars": r"\bjars?\b",
        "plastic containers": r"\bplastic containers?\b"
    }

    found = [
        name for name, pattern in patterns.items()
        if re.search(pattern, text)
    ]

    return ", ".join(found) if found else None


df_market_prices["Packaging"] = (
    df_market_prices["OfferContext"].apply(extract_packaging)
)

df_market_prices["HasPackagingInfo"] = (
    df_market_prices["Packaging"].notna()
)

display(
    df_market_prices[
        ["Product", "OfferContext", "Packaging", "HasPackagingInfo"]
    ].head(20)
)

,Product,OfferContext,Packaging,HasPackagingInfo
0,ANISE,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,cartons,True
1,ANISE,24s,None,False
2,ANISE,fine appearance,None,False
3,ARUGULA,MARKET STEADY. cartons bunched NEW JERSEY 24s,cartons,True
4,ARUGULA,with root 4/5 bushel crates bunched FLORIDA 24s,crates,True
5,ARUGULA,with root cartons 2 1-1/2 lb film bags FLORIDA...,"cartons, bags",True
6,BASIL,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...,cartons,True
7,BASIL,bushel cartons bunched FLORIDA WITH ROOT 15s,cartons,True
8,BAY LEAVES,OFFERINGS LIGHT. repacked enroute,None,False
9,BAY LEAVES,1 KG cartons film bag loose 1 lb film bags loo...,"cartons, bags",True


In [180]:
df_market_prices["HasBushel"] = (
    df_market_prices["OfferContext"].str.contains(
        r"\bbushel\b", case=False, na=False
    )
)

In [181]:
packaging_stats = pd.DataFrame({
    "Заполнено": [
        df_market_prices["Packaging"].notna().sum(),
        df_market_prices["HasBushel"].sum()
    ],
    "Не заполнено": [
        df_market_prices["Packaging"].isna().sum(),
        (~df_market_prices["HasBushel"]).sum()
    ]
}, index=["Тип упаковки", "Указан bushel"])

packaging_stats["Доля заполнения, %"] = (
    packaging_stats["Заполнено"] /
    len(df_market_prices) * 100
).round(2)

display(packaging_stats)

,Заполнено,Не заполнено,"Доля заполнения, %"
Тип упаковки,309,659,31.92
Указан bushel,69,899,7.13


In [182]:
locations = [
    "CALIFORNIA", "FLORIDA", "TEXAS", "NEW YORK",
    "NEW JERSEY", "MICHIGAN", "CANADA", "MEXICO",
    "COLOMBIA", "CHILE", "PERU", "BRAZIL",
    "ARGENTINA", "GUATEMALA", "HONDURAS"
]

for location in locations:
    count = df_market_prices["OfferContext"].str.contains(
        rf"\b{location}\b",
        case=False,
        na=False
    ).sum()

    print(f"{location}: {count}")

CALIFORNIA: 57
FLORIDA: 23
TEXAS: 3
NEW YORK: 27
NEW JERSEY: 37
MICHIGAN: 5
CANADA: 51
MEXICO: 48
COLOMBIA: 16
CHILE: 8
PERU: 9
BRAZIL: 3
ARGENTINA: 1
GUATEMALA: 9
HONDURAS: 2


In [183]:
locations = [
    "CALIFORNIA", "FLORIDA", "TEXAS", "NEW YORK",
    "NEW JERSEY", "MICHIGAN", "CANADA", "MEXICO",
    "COLOMBIA", "CHILE", "PERU", "BRAZIL",
    "ARGENTINA", "GUATEMALA", "HONDURAS"
]

def extract_origin(text):
    found = []

    for location in locations:
        if re.search(rf"\b{re.escape(location)}\b", text, re.IGNORECASE):
            found.append(location.title())

    return ", ".join(found) if found else None


df_market_prices["Origin"] = (
    df_market_prices["OfferContext"].apply(extract_origin)
)

df_market_prices["HasOriginInfo"] = (
    df_market_prices["Origin"].notna()
)

display(
    df_market_prices[
        ["Product", "OfferContext", "Origin"]
    ].head(20)
)

,Product,OfferContext,Origin
0,ANISE,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,California
1,ANISE,24s,None
2,ANISE,fine appearance,None
3,ARUGULA,MARKET STEADY. cartons bunched NEW JERSEY 24s,New Jersey
4,ARUGULA,with root 4/5 bushel crates bunched FLORIDA 24s,Florida
5,ARUGULA,with root cartons 2 1-1/2 lb film bags FLORIDA...,Florida
6,BASIL,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...,Colombia
7,BASIL,bushel cartons bunched FLORIDA WITH ROOT 15s,Florida
8,BAY LEAVES,OFFERINGS LIGHT. repacked enroute,None
9,BAY LEAVES,1 KG cartons film bag loose 1 lb film bags loo...,None


In [184]:
origin_stats = pd.DataFrame({
    "Заполнено": [
        df_market_prices["Origin"].notna().sum()
    ],
    "Не заполнено": [
        df_market_prices["Origin"].isna().sum()
    ]
}, index=["Происхождение"])

origin_stats["Доля заполнения, %"] = (
    origin_stats["Заполнено"] /
    len(df_market_prices) * 100
).round(2)

display(origin_stats)

,Заполнено,Не заполнено,"Доля заполнения, %"
Происхождение,297,671,30.68


In [185]:
measurement_patterns = {
    "lb": r"\b\d+(?:\.\d+)?\s*lb\b",
    "kg": r"\b\d+(?:\.\d+)?\s*kg\b",
    "oz": r"\b\d+(?:\.\d+)?\s*oz\b",
    "gram": r"\b\d+(?:\.\d+)?\s*g\b",
    "dozen": r"\b\d+\s*(?:dozen|dz)\b",
    "count": r"\b\d+s\b"
}

for name, pattern in measurement_patterns.items():
    count = df_market_prices["OfferContext"].str.contains(
        pattern, case=False, na=False
    ).sum()

    print(f"{name}: {count}")

lb: 108
kg: 33
oz: 0
gram: 0
dozen: 5
count: 431


In [186]:
def extract_measurement(text):
    result = {
        "WeightValue": None,
        "WeightUnit": None,
        "CountValue": None,
        "DozenValue": None
    }

    # Масса: lb или kg
    weight_match = re.search(
        r"\b(\d+(?:\.\d+)?)\s*(lb|kg)\b",
        text,
        re.IGNORECASE
    )

    if weight_match:
        result["WeightValue"] = float(weight_match.group(1))
        result["WeightUnit"] = weight_match.group(2).lower()

    # Количество штук, например 24s
    count_match = re.search(
        r"\b(\d+)s\b",
        text,
        re.IGNORECASE
    )

    if count_match:
        result["CountValue"] = int(count_match.group(1))

    # Количество дюжин
    dozen_match = re.search(
        r"\b(\d+)\s*(?:dozen|dz)\b",
        text,
        re.IGNORECASE
    )

    if dozen_match:
        result["DozenValue"] = int(dozen_match.group(1))

    return pd.Series(result)


df_market_prices[
    ["WeightValue", "WeightUnit", "CountValue", "DozenValue"]
] = df_market_prices["OfferContext"].apply(extract_measurement)

display(
    df_market_prices[
        [
            "Product",
            "OfferContext",
            "WeightValue",
            "WeightUnit",
            "CountValue",
            "DozenValue"
        ]
    ].head(30)
)

,Product,OfferContext,WeightValue,WeightUnit,CountValue,DozenValue
0,ANISE,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,NaN,NaN,18.0,NaN
1,ANISE,24s,NaN,NaN,24.0,NaN
2,ANISE,fine appearance,NaN,None,NaN,NaN
3,ARUGULA,MARKET STEADY. cartons bunched NEW JERSEY 24s,NaN,NaN,24.0,NaN
4,ARUGULA,with root 4/5 bushel crates bunched FLORIDA 24s,NaN,NaN,24.0,NaN
5,ARUGULA,with root cartons 2 1-1/2 lb film bags FLORIDA...,2.0,lb,NaN,NaN
6,BASIL,OFFERINGS LIGHT. 1 kg cartons bunched COLOMBIA...,1.0,kg,NaN,NaN
7,BASIL,bushel cartons bunched FLORIDA WITH ROOT 15s,NaN,NaN,15.0,NaN
8,BAY LEAVES,OFFERINGS LIGHT. repacked enroute,NaN,None,NaN,NaN
9,BAY LEAVES,1 KG cartons film bag loose 1 lb film bags loo...,1.0,kg,NaN,NaN


In [187]:
weight_pattern = re.compile(
    r"(?<![\w.])(\d+(?:\.\d+)?(?:\s*-\s*\d+(?:\.\d+)?)?)\s*-?\s*(lb|kg)\b",
    re.IGNORECASE
)


def extract_all_weights(text):
    matches = weight_pattern.findall(text)

    return [
        {
            "Value": value.strip(),
            "Unit": unit.lower()
        }
        for value, unit in matches
    ]

df_market_prices["AllWeights"] = (
    df_market_prices["OfferContext"].apply(extract_all_weights)
)

display(
    df_market_prices.loc[
        df_market_prices["AllWeights"].apply(len) > 1,
        ["Product", "OfferContext", "AllWeights"]
    ].head(30)
)

,Product,OfferContext,AllWeights
9,BAY LEAVES,1 KG cartons film bag loose 1 lb film bags loo...,"[{'Value': '1', 'Unit': 'kg'}, {'Value': '1', ..."
21,CIPOLINOS,5 lb jar 10 lb mesh sacks NEW YORK ORANGE COUN...,"[{'Value': '5', 'Unit': 'lb'}, {'Value': '10',..."
29,LEMON GRASS,5-lb 40 lb cartons CALIFORNIA,"[{'Value': '5', 'Unit': 'lb'}, {'Value': '40',..."
100,CARROTS,5lb. film bag cartons 30 1-lb film bags CALIFO...,"[{'Value': '5', 'Unit': 'lb'}, {'Value': '1', ..."
146,EGGPLANT,10 lb. carton 5 kg/11 lb cartons NETHERLANDS g...,"[{'Value': '10', 'Unit': 'lb'}, {'Value': '5',..."
147,EGGPLANT,5 kg/11 lb cartons tray pack CANADA ONTARIO IT...,"[{'Value': '5', 'Unit': 'kg'}, {'Value': '11',..."
298,"PEPPERS, BELL TYPE",5 kg/11 lb cartons CANADA ONTARIO greenhouse O...,"[{'Value': '5', 'Unit': 'kg'}, {'Value': '11',..."
407,TOMATOES,occasional higher 5 kg/11 lb flats CANADA ONTA...,"[{'Value': '5', 'Unit': 'kg'}, {'Value': '11',..."
817,KIWIFRUIT,9 kg (19.8 lb) containers loose CHILE HAYWARD ...,"[{'Value': '9', 'Unit': 'kg'}, {'Value': '19.8..."


In [188]:
def classify_weights(weights):
    if len(weights) == 0:
        return "Масса не указана"

    if len(weights) == 1:
        return "Одно значение массы"

    # Проверяем, есть ли значения в разных единицах
    units = {item["Unit"] for item in weights}

    if len(units) > 1:
        return "Несколько значений массы"

    return "Несколько значений в одной единице"


df_market_prices["WeightGroup"] = (
    df_market_prices["AllWeights"].apply(classify_weights)
)

display(df_market_prices["WeightGroup"].value_counts())

WeightGroup
Масса не указана                      807
Одно значение массы                   152
Несколько значений массы                6
Несколько значений в одной единице      3
Name: count, dtype: int64

In [189]:
# Размер датасета
print("Размер датасета:", df_market_prices.shape)

# Названия и типы столбцов
print("\nИнформация о столбцах:")
df_market_prices.info()

# Первые записи
display(df_market_prices.head())

# Количество уникальных товаров
print("\nУникальных товаров:", df_market_prices["Product"].nunique())

# Количество уникальных дат отчётов
print("Дат отчётов:", df_market_prices["ReportDate"].nunique())

# Период наблюдений
print("Начало периода:", df_market_prices["ReportDate"].min())
print("Конец периода:", df_market_prices["ReportDate"].max())

Размер датасета: (968, 24)

Информация о столбцах:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Dataset              968 non-null    object        
 1   Page                 968 non-null    int64         
 2   ReportDate           968 non-null    datetime64[ns]
 3   Section              244 non-null    object        
 4   Product              968 non-null    object        
 5   OfferNumber          968 non-null    int64         
 6   OfferContext         968 non-null    object        
 7   PriceMin             968 non-null    float64       
 8   PriceMax             968 non-null    float64       
 9   MostlyMin            98 non-null     float64       
 10  MostlyMax            98 non-null     float64       
 11  PriceType            968 non-null    object        
 12  OriginalDescription  968 non-null    obje

,Dataset,Page,ReportDate,Section,Product,OfferNumber,OfferContext,PriceMin,PriceMax,MostlyMin,...,HasPackagingInfo,HasBushel,Origin,HasOriginInfo,WeightValue,WeightUnit,CountValue,DozenValue,AllWeights,WeightGroup
0,Овощи,1,2026-11-06,HERBS,ANISE,1,MARKET ABOUT STEADY. cartons CALIFORNIA 18s,24.0,26.0,NaN,...,True,False,California,True,NaN,NaN,18.0,NaN,[],Масса не указана
1,Овощи,1,2026-11-06,HERBS,ANISE,2,24s,22.0,24.0,NaN,...,False,False,None,False,NaN,NaN,24.0,NaN,[],Масса не указана
2,Овощи,1,2026-11-06,HERBS,ANISE,3,fine appearance,26.0,30.0,28.0,...,False,False,None,False,NaN,None,NaN,NaN,[],Масса не указана
3,Овощи,1,2026-11-06,HERBS,ARUGULA,1,MARKET STEADY. cartons bunched NEW JERSEY 24s,22.0,24.0,NaN,...,True,False,New Jersey,True,NaN,NaN,24.0,NaN,[],Масса не указана
4,Овощи,1,2026-11-06,HERBS,ARUGULA,2,with root 4/5 bushel crates bunched FLORIDA 24s,24.0,24.0,NaN,...,True,True,Florida,True,NaN,NaN,24.0,NaN,[],Масса не указана



Уникальных товаров: 177
Дат отчётов: 1
Начало периода: 2026-11-06 00:00:00
Конец периода: 2026-11-06 00:00:00


In [190]:
missing_stats = pd.DataFrame({
    "Заполнено": df_market_prices.notna().sum(),
    "Пропущено": df_market_prices.isna().sum(),
    "Доля заполнения, %": (
        df_market_prices.notna().mean() * 100
    ).round(2),
    "Доля пропусков, %": (
        df_market_prices.isna().mean() * 100
    ).round(2)
})

display(missing_stats)

,Заполнено,Пропущено,"Доля заполнения, %","Доля пропусков, %"
Dataset,968,0,100.00,0.00
Page,968,0,100.00,0.00
ReportDate,968,0,100.00,0.00
Section,244,724,25.21,74.79
Product,968,0,100.00,0.00
OfferNumber,968,0,100.00,0.00
OfferContext,968,0,100.00,0.00
PriceMin,968,0,100.00,0.00
PriceMax,968,0,100.00,0.00
MostlyMin,98,870,10.12,89.88


In [191]:
# Исключаем столбцы со списками и словарями
duplicate_columns = [
    col for col in df_market_prices.columns
    if col != "AllWeights"
]

# Полные дубликаты по остальным столбцам
print(
    "Полных дубликатов:",
    df_market_prices.duplicated(subset=duplicate_columns).sum()
)

# Дубликаты по ключевым характеристикам
key_columns = [
    "Dataset",
    "ReportDate",
    "Product",
    "OfferContext",
    "PriceMin",
    "PriceMax"
]

print(
    "Дубликатов по ключевым характеристикам:",
    df_market_prices.duplicated(subset=key_columns).sum()
)

Полных дубликатов: 0
Дубликатов по ключевым характеристикам: 32


In [192]:
print("Нулевые минимальные цены:",
      (df_market_prices["PriceMin"] == 0).sum())

print("Отрицательные минимальные цены:",
      (df_market_prices["PriceMin"] < 0).sum())

print("Отрицательные максимальные цены:",
      (df_market_prices["PriceMax"] < 0).sum())

print("Минимальная цена:", df_market_prices["PriceMin"].min())
print("Максимальная цена:", df_market_prices["PriceMax"].max())

# Проверка случаев, когда минимальная цена выше максимальной
invalid_ranges = df_market_prices[
    df_market_prices["PriceMin"] > df_market_prices["PriceMax"]
]

print("Некорректных ценовых диапазонов:", len(invalid_ranges))
display(invalid_ranges)

Нулевые минимальные цены: 0
Отрицательные минимальные цены: 0
Отрицательные максимальные цены: 0
Минимальная цена: 1.35
Максимальная цена: 225.0
Некорректных ценовых диапазонов: 0


,Dataset,Page,ReportDate,Section,Product,OfferNumber,OfferContext,PriceMin,PriceMax,MostlyMin,...,HasPackagingInfo,HasBushel,Origin,HasOriginInfo,WeightValue,WeightUnit,CountValue,DozenValue,AllWeights,WeightGroup


In [193]:
# Заполненность разделов
print("Распределение разделов:")
display(
    df_market_prices["Section"]
    .value_counts(dropna=False)
)

# Количество предложений по товарам
product_stats = (
    df_market_prices.groupby("Product")
    .agg(
        Количество_предложений=("PriceMin", "count"),
        Минимальная_цена=("PriceMin", "min"),
        Максимальная_цена=("PriceMax", "max")
    )
    .sort_values("Количество_предложений", ascending=False)
)

display(product_stats.head(20))

# Описательная статистика цен
display(
    df_market_prices[["PriceMin", "PriceMax", "MostlyMin", "MostlyMax"]]
    .describe()
)

Распределение разделов:


Section
None                  724
CITRUS                 78
VEGETABLES             68
HERBS                  45
MELONS                 20
BERRIES                17
ORGANIC BERRIES         7
OTHER FRUIT             5
ORGANIC VEGETABLES      3
ORGANIC CITRUS          1
Name: count, dtype: int64

,Количество_предложений,Минимальная_цена,Максимальная_цена
Product,,,
APPLES,176,20.0,72.0
MUSHROOMS,41,9.5,40.0
ORANGES,35,25.0,45.0
PEARS,31,28.0,55.0
PERSIMMONS,24,15.0,36.0
KIWIFRUIT,22,18.0,48.0
MANGOES,21,6.0,13.0
LEMONS,19,24.0,48.0
GRAPES,15,24.0,50.0


,PriceMin,PriceMax,MostlyMin,MostlyMax
count,968.000000,968.000000,98.000000,98.000000
mean,34.079907,35.721436,34.500000,34.765306
std,23.026767,23.921094,16.825514,17.218308
min,1.350000,1.350000,11.000000,11.000000
25%,22.000000,23.000000,26.000000,26.000000
50%,30.000000,32.000000,31.500000,32.000000
75%,40.000000,42.000000,39.500000,40.000000
max,225.000000,225.000000,114.000000,118.000000


In [194]:
def find_price_outliers(group):
    q1 = group["PriceMin"].quantile(0.25)
    q3 = group["PriceMin"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[
        (group["PriceMin"] < lower) |
        (group["PriceMin"] > upper)
    ]

price_outliers = (
    df_market_prices.groupby("Product", group_keys=False)
    .apply(find_price_outliers, include_groups=False)
)

print("Количество потенциальных выбросов:", len(price_outliers))

Количество потенциальных выбросов: 36


In [195]:
outlier_records = []

for product, group in df_market_prices.groupby("Product"):
    q1 = group["PriceMin"].quantile(0.25)
    q3 = group["PriceMin"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = group[
        (group["PriceMin"] < lower) |
        (group["PriceMin"] > upper)
    ].copy()

    outlier_records.append(outliers)

price_outliers = pd.concat(outlier_records)

print("Количество потенциальных выбросов:", len(price_outliers))

display(
    price_outliers[
        [
            "Product",
            "OfferContext",
            "PriceMin",
            "PriceMax",
            "MostlyMin",
            "MostlyMax",
            "Packaging",
            "Origin",
            "AllWeights",
            "OriginalDescription"
        ]
    ].sort_values(["Product", "PriceMin"])
)

Количество потенциальных выбросов: 36


,Product,OfferContext,PriceMin,PriceMax,MostlyMin,MostlyMax,Packaging,Origin,AllWeights,OriginalDescription
722,APPLES,HONEYCRISP 72s,58.0,60.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
723,APPLES,80s,62.0,64.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
725,APPLES,PINK LADY/CRIPPS PINK 80s,62.0,62.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
587,APPLES,cartons tray pack CANADA CD EXTRA FANCY HONEYC...,64.0,64.0,NaN,NaN,cartons,Canada,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
591,APPLES,HONEYCRISP 72s fine appearance,65.0,65.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
592,APPLES,80s fine appearance,65.0,65.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
703,APPLES,fine appearance,66.0,68.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
701,APPLES,72s fine appearance,70.0,70.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
726,APPLES,fine appearance,70.0,72.0,NaN,NaN,None,None,[],MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
966,BANANAS,MARKET ABOUT STEADY. 40 lb cartons bagged COLO...,30.0,32.0,NaN,NaN,cartons,Colombia,"[{'Value': '40', 'Unit': 'lb'}]",MARKET ABOUT STEADY. 40 lb cartons bagged COLO...


In [196]:
display(
    df_market_prices[
        df_market_prices["Product"] == "APPLES"
    ][
        [
            "OfferContext",
            "PriceMin",
            "PriceMax",
            "Packaging",
            "Origin",
            "AllWeights"
        ]
    ].head(30)
)

,OfferContext,PriceMin,PriceMax,Packaging,Origin,AllWeights
565,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...,34.0,34.0,"cartons, bags",Canada,"[{'Value': '3', 'Unit': 'lb'}]"
566,"NEW YORK U.S. EXTRA FANCY EMPIRE 2 1/2"" minimum",26.0,28.0,None,New York,[]
567,"FUJI 2 1/4"" minimum",26.0,28.0,None,None,[]
568,"GALA 2 1/2"" minimum",30.0,30.0,None,None,[]
569,"2 1/4"" minimum",26.0,28.0,None,None,[]
570,"GOLDEN DELICIOUS 2 1/2"" minimum",30.0,30.0,None,None,[]
571,"2 1/4"" minimum",24.0,24.0,None,None,[]
572,"HONEYCRISP 2 1/2"" minimum",32.0,32.0,None,None,[]
573,"2 1/4"" minimum",28.0,28.0,None,None,[]
574,"MCINTOSH 2 1/2"" minimum",28.0,30.0,None,None,[]


In [197]:
df_market_prices.loc[
    [443, 964],
    ["Product", "OfferContext", "OriginalDescription", "AllWeights"]
].to_string(index=False)

"     Product                                                                                                                      OfferContext                                                                                                                                                                                                                                                                                                         OriginalDescription                     AllWeights\nSTRAWBERRIES                       MARKET MEXICO SLIGHTLY LOWER, OTHERS ABOUT STEADY. flats 8 1-lb containers with lids CALIFORNIA extra large MARKET MEXICO SLIGHTLY LOWER, OTHERS ABOUT STEADY. flats 8 1-lb containers with lids CALIFORNIA extra large 40.00-44.00 mostly 42.00 fine appearance 48.00-52.00 mostly 50.00 MEXICO extra large 28.00-32.00 mostly 30.00 holdovers 22.00- 24.00 flats 4 2-lb containers with lids MEXICO extra large fine appearance 42.00 [{'Value': '1', 'Unit': 'lb'}]\nSTRAWBERRIES MARK

In [198]:
duplicates = df_market_prices[
    df_market_prices.duplicated(
        subset=key_columns,
        keep=False
    )
].sort_values(key_columns)

duplicates[
    [
        "Dataset", "Page", "Product",
        "OfferContext", "PriceMin", "PriceMax",
        "OriginalDescription"
    ]
]

,Dataset,Page,Product,OfferContext,PriceMin,PriceMax,OriginalDescription
91,Овощи,2,CABBAGE,ROUND GREEN TYPE large,16.0,18.0,MARKET ABOUT STEADY. 50 lb cartons CANADA QUEB...
93,Овощи,2,CABBAGE,ROUND GREEN TYPE large,16.0,18.0,MARKET ABOUT STEADY. 50 lb cartons CANADA QUEB...
640,Фрукты,3,APPLES,100s,28.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
670,Фрукты,3,APPLES,100s,28.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
629,Фрукты,3,APPLES,100s,30.0,30.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
658,Фрукты,3,APPLES,100s,30.0,30.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
650,Фрукты,3,APPLES,138s,28.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
660,Фрукты,3,APPLES,138s,28.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
569,Фрукты,3,APPLES,"2 1/4"" minimum",26.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...
578,Фрукты,3,APPLES,"2 1/4"" minimum",26.0,28.0,MARKET SLIGHTLY LOWER. cartons 12 3-lb film ba...


In [199]:
duplicate_groups = (
    duplicates
    .groupby(key_columns, dropna=False)["OriginalDescription"]
    .nunique()
)

print("Групп с различающимися описаниями:", (duplicate_groups > 1).sum())

Групп с различающимися описаниями: 0


In [200]:
df_market_prices = df_market_prices.drop_duplicates(
    subset=key_columns,
    keep="first"
).reset_index(drop=True)

print("Строк после удаления:", len(df_market_prices))

Строк после удаления: 936


In [201]:
sections = {
    "Овощи": [
        "HERBS",
        "VEGETABLES",
        "ORGANIC VEGETABLES"
    ],
    "Фрукты": [
        "CITRUS",
        "MELONS",
        "OTHER FRUIT",
        "BERRIES",
        "ORGANIC BERRIES",
        "ORGANIC CITRUS"
    ]
}

df_market_records["Section"] = None

for dataset in ["Овощи", "Фрукты"]:
    current_section = None

    source = df_market[
        df_market["Dataset"] == dataset
    ].sort_values(["Page", "LineNumber"])

    for _, line in source.iterrows():
        text = line["CleanText"].strip()

        # Проверяем, является ли строка заголовком раздела
        if text in sections[dataset]:
            current_section = text
            continue

        # Если строка начинается с обозначения товара,
        # присваиваем ему текущую категорию
        if text.startswith("---"):
            product = text.split(":", 1)[0].replace("---", "").strip()

            mask = (
                (df_market_records["Dataset"] == dataset) &
                (df_market_records["Page"] == line["Page"]) &
                (df_market_records["Product"] == product)
            )

            df_market_records.loc[mask, "Section"] = current_section

In [202]:
print("Распределение категорий:")
print(df_market_records["Section"].value_counts(dropna=False))

print(
    "\nКоличество пропусков:",
    df_market_records["Section"].isna().sum()
)

print("\nПримеры оставшихся пропусков:")
display(
    df_market_records[
        df_market_records["Section"].isna()
    ][["Dataset", "Page", "Product", "Description"]].head(20)
)

Распределение категорий:
Section
VEGETABLES            102
OTHER FRUIT            35
HERBS                  22
CITRUS                  9
BERRIES                 5
MELONS                  3
ORGANIC BERRIES         3
ORGANIC CITRUS          2
ORGANIC VEGETABLES      1
Name: count, dtype: int64

Количество пропусков: 0

Примеры оставшихся пропусков:


,Dataset,Page,Product,Description


In [203]:
print("Количество товарных записей:", len(df_market_records))
print("Количество ценовых предложений:", len(df_market_prices))
print("Количество уникальных товаров:", df_market_records["Product"].nunique())
print("Количество товаров в таблице цен:", df_market_prices["Product"].nunique())

# Проверяем, все ли товары из таблицы цен есть в таблице товарных записей
missing_products = set(df_market_prices["Product"]) - set(df_market_records["Product"])

print("Товаров без исходной записи:", len(missing_products))
print(missing_products)

Количество товарных записей: 182
Количество ценовых предложений: 936
Количество уникальных товаров: 177
Количество товаров в таблице цен: 177
Товаров без исходной записи: 0
set()


In [204]:
print(
    "Пустые исходные описания:",
    df_market_prices["OriginalDescription"].isna().sum()
)

print(
    "Пустые контексты предложений:",
    df_market_prices["OfferContext"].isna().sum()
)

Пустые исходные описания: 0
Пустые контексты предложений: 0


In [205]:
section_map = df_market_records[
    ["Dataset", "Page", "Product", "Section"]
].drop_duplicates(
    subset=["Dataset", "Page", "Product"]
)

df_market_prices = df_market_prices.drop(
    columns=["Section"]
).merge(
    section_map,
    on=["Dataset", "Page", "Product"],
    how="left",
    validate="many_to_one"
)

print("Пропуски в категориях:", df_market_prices["Section"].isna().sum())
print(df_market_prices["Section"].value_counts())

Пропуски в категориях: 0
Section
VEGETABLES            377
OTHER FRUIT           370
CITRUS                 89
HERBS                  50
MELONS                 20
BERRIES                17
ORGANIC BERRIES         7
ORGANIC VEGETABLES      3
ORGANIC CITRUS          3
Name: count, dtype: int64


In [206]:
import os

output_dir = "processed_datasets/USDA_AMS_MyMarketNews"
os.makedirs(output_dir, exist_ok=True)

# CSV для просмотра и дальнейшего использования
df_market_prices.to_csv(
    os.path.join(output_dir, "USDA_AMS_MyMarketNews_Clean.csv"),
    index=False,
    encoding="utf-8-sig"
)

# PKL для сохранения структуры и типов данных
df_market_prices.to_pickle(
    os.path.join(output_dir, "USDA_AMS_MyMarketNews_Clean.pkl")
)

print("Количество записей:", len(df_market_prices))
print("Количество столбцов:", len(df_market_prices.columns))
print("Количество уникальных продуктов:", df_market_prices["Product"].nunique())
print("Файлы сохранены в:", output_dir)

Количество записей: 936
Количество столбцов: 24
Количество уникальных продуктов: 177
Файлы сохранены в: processed_datasets/USDA_AMS_MyMarketNews
